# K-means con RDDs

Primero implementamos el algoritmo a mano usando transformaciones de RDD, y después lo comparamos con la versión que ya trae MLlib.

In [1]:
# Instalar SDK Java (25 si está disponible, si no 21)
!sudo apt-get update -qq > /dev/null
!sudo apt-get install -y openjdk-25-jdk-headless -qq > /dev/null 2>&1 || \
 sudo apt-get install -y openjdk-21-jdk-headless -qq > /dev/null

# Descargar Spark 4.2.0
!wget -q https://archive.apache.org/dist/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
# Descomprimir el archivo descargado
!tar xf spark-4.2.0-bin-hadoop3.tgz

# Configurar variables de entorno
import os
os.environ["SPARK_HOME"] = "/content/spark-4.2.0-bin-hadoop3"

# JAVA_HOME apunta al JDK que realmente se haya instalado
for version in (25, 21, 17):
    ruta = f"/usr/lib/jvm/java-{version}-openjdk-amd64"
    if os.path.isdir(ruta):
        os.environ["JAVA_HOME"] = ruta
        break

# Desinstalar dataproc-spark-connect
!pip uninstall -y -q dataproc-spark-connect
# Instalar findspark
!pip install -q findspark
# Instalar pyspark
!pip install -q pyspark==4.2.0

# Se importa la libreria findspark
import findspark
findspark.init()

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
JAVA_HOME: /usr/lib/jvm/java-25-openjdk-amd64


## Implementación manual

K-means repite dos pasos hasta que los centroides dejan de moverse:

1. Asignar cada punto al centroide más cercano.
2. Recalcular cada centroide como el promedio de sus puntos.

El primer paso es un `map` y el segundo un `reduceByKey`.

In [2]:
from pyspark import SparkConf, SparkContext
import numpy as np

conf = SparkConf().setAppName("KMeans").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf)

# Puntos en 2D
datos = np.array([
    [2.0, 3.0], [10.0, 15.0], [8.0, 8.0],
    [7.0, 5.0], [3.0, 7.0], [11.0, 14.0],
    [6.0, 5.0]
])

rdd = sc.parallelize(datos)

k = 2
max_iteraciones = 15

# Usamos los primeros k puntos como centroides iniciales.
# El copy() es importante: sin él modificaríamos 'datos' al actualizarlos.
centroides = datos[:k].copy()

print(centroides)

[[ 2.  3.]
 [10. 15.]]


In [3]:
def centroide_mas_cercano(punto, centroides):
    """Devuelve el índice del centroide más cercano al punto."""
    distancias = np.sqrt(np.sum((centroides - punto) ** 2, axis=1))
    return np.argmin(distancias)

Cada elemento del RDD se convierte en un par `(cluster, (punto, 1))`. El `1` nos sirve para ir contando cuántos puntos lleva cada grupo y poder sacar el promedio al final.

In [4]:
for iteracion in range(max_iteraciones):
    # Paso 1: asignar cada punto a un cluster
    asignaciones = rdd.map(lambda punto: (centroide_mas_cercano(punto, centroides), (punto, 1)))

    # Paso 2: sumar los puntos de cada cluster y contarlos
    sumas = asignaciones.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

    # El nuevo centroide es el promedio del cluster
    nuevos_centroides = sumas.map(lambda x: (x[0], x[1][0] / x[1][1])).collect()

    for indice, centroide in nuevos_centroides:
        centroides[indice] = centroide

In [5]:
for i, centroide in enumerate(centroides):
    print(f"Centroide {i}: {centroide}")

Centroide 0: [5.2 5.6]
Centroide 1: [10.5 14.5]


## La versión de MLlib

`pyspark.mllib` es la API de machine learning basada en RDDs. Hoy está en modo mantenimiento; para proyectos nuevos se usa `pyspark.ml`, que trabaja sobre DataFrames. La usamos aquí porque encaja con el tema de la sesión.

In [6]:
from pyspark.mllib.clustering import KMeans

modelo = KMeans.train(rdd, k, maxIterations=max_iteraciones)

for i, centroide in enumerate(modelo.clusterCenters):
    print(f"Centroide {i}: {centroide}")

Centroide 0: [5.2 5.6]
Centroide 1: [10.5 14.5]


Los centroides coinciden con los que calculamos a mano, pero pueden salir en otro orden. El número de cluster es solo una etiqueta: no significa nada por sí mismo.

In [7]:
# Predecir el cluster de un punto nuevo
punto_nuevo = np.array([7.0, 10.0])

print("El punto", punto_nuevo, "cae en el cluster", modelo.predict(punto_nuevo))

El punto [ 7. 10.] cae en el cluster 0


## Ejercicios

**1.** Predice a qué cluster pertenecen los puntos `[1.0, 1.0]`, `[9.0, 9.0]` y `[12.0, 16.0]`.

**2.** Entrena un nuevo modelo con `k = 3` y muestra los centroides. ¿Qué grupo se partió en dos?

In [9]:
# Ejercicio 1

puntos_nuevos = [
    np.array([1.0, 1.0]),
    np.array([9.0, 9.0]),
    np.array([12.0, 16.0])
]

for punto in puntos_nuevos:
    cluster = modelo.predict(punto)
    print(f"El punto {punto} pertenece al cluster {cluster}")

El punto [1. 1.] pertenece al cluster 0
El punto [9. 9.] pertenece al cluster 0
El punto [12. 16.] pertenece al cluster 1


In [26]:
# Ejercico 2
k = 3

modelo_k3 = KMeans.train(
    rdd,
    k,
    maxIterations=max_iteraciones
)

for i, centroide in enumerate(modelo_k3.clusterCenters):
    print(f"Centroide {i}: {centroide}")

Centroide 0: [2.5 5. ]
Centroide 1: [10.5 14.5]
Centroide 2: [7. 6.]


In [19]:
# Modelo con k=2

for punto in datos:
    cluster = modelo.predict(punto)
    print(f"{punto} -> cluster {cluster}")

[2. 3.] -> cluster 0
[10. 15.] -> cluster 1
[8. 8.] -> cluster 0
[7. 5.] -> cluster 0
[3. 7.] -> cluster 0
[11. 14.] -> cluster 1
[6. 5.] -> cluster 0


In [28]:
# Modelo con k = 3

for punto in datos:
    cluster = modelo_k3.predict(punto)
    print(f"{punto} -> cluster {cluster}")

# El grupo que contenía [10,15], [8. 8.], [7. 5.], [3. 7.], [6. 5.]   se partió en dos.

[2. 3.] -> cluster 0
[10. 15.] -> cluster 1
[8. 8.] -> cluster 2
[7. 5.] -> cluster 2
[3. 7.] -> cluster 0
[11. 14.] -> cluster 1
[6. 5.] -> cluster 2


In [29]:
# Liberar los recursos
sc.stop()